# FX data-quality audit — 2026-09-25

Purpose: decide whether stored cash USD can translate TSE stock prices, and whether USDT can substitute for it. Grain is one `(symbol, Jalali date)` in `marketdata_goldcurrencyhistory`. This notebook reads only public market rows from the portfolio VPS through the existing SSH connection and does not print credentials. Results are a dated snapshot, not a live certification. Execute with a Python environment containing `jdatetime` and SSH access to `45.139.10.12`. Source model: `backend/marketdata/models.py::GoldCurrencyHistory`.


In [ ]:
from collections import defaultdict
from datetime import timedelta
from decimal import Decimal
import re
import subprocess

import jdatetime

HOST = "45.139.10.12"
PSQL = "docker exec -i portfolio-saas-db-1 psql -U portfolio -d portfolio -At -F '|'"

def query(sql):
    result = subprocess.run(
        ["ssh", "-o", "BatchMode=yes", HOST, PSQL],
        input=sql, text=True, capture_output=True, check=True,
    )
    return [line.split("|") for line in result.stdout.splitlines() if line]


In [ ]:
rows = query('''
SELECT symbol, date, close_price, unit, source,
       ingested_at IS NOT NULL, last_correlation_id <> ''
FROM marketdata_goldcurrencyhistory
WHERE symbol IN ('USD', 'USDT_IRT')
ORDER BY symbol, date;
''')
series = defaultdict(list)
for symbol, day, price, unit, source, stamped, lineage in rows:
    assert re.fullmatch(r"1[34]\d{2}-\d{2}-\d{2}", day)
    assert unit == "تومان" and source == "provider" and Decimal(price) > 0
    year, month, date = map(int, day.split("-"))
    series[symbol].append({
        "day": day, "gregorian": jdatetime.date(year, month, date).togregorian(),
        "price": Decimal(price), "ingested_at": stamped == "t", "lineage": lineage == "t",
    })
for symbol, records in series.items():
    print(symbol, "rows", len(records), "range", records[0]["day"], records[-1]["day"],
          "missing ingestion stamps", sum(not r["ingested_at"] for r in records),
          "missing lineage", sum(not r["lineage"] for r in records))


In [ ]:
for symbol, records in series.items():
    observed = {r["gregorian"] for r in records}
    day = records[0]["gregorian"]
    missing = []
    while day <= records[-1]["gregorian"]:
        if day not in observed:
            missing.append(day.isoformat())
        day += timedelta(days=1)
    runs = []
    start = 0
    for index in range(1, len(records) + 1):
        if index == len(records) or records[index]["price"] != records[start]["price"]:
            runs.append((index - start, records[start]["day"], records[index - 1]["day"]))
            start = index
    print(symbol, "missing calendar days", len(missing), "first 20", missing[:20],
          "longest unchanged-price run", max(runs))


In [ ]:
checks = query('''
SELECT 'overlap', left(a.date,4), count(*)::text,
       round(percentile_cont(0.5) WITHIN GROUP
         (ORDER BY abs(a.close_price-b.close_price)/a.close_price)::numeric * 100,2)::text,
       round(percentile_cont(0.9) WITHIN GROUP
         (ORDER BY abs(a.close_price-b.close_price)/a.close_price)::numeric * 100,2)::text
FROM marketdata_goldcurrencyhistory a
JOIN marketdata_goldcurrencyhistory b ON b.date=a.date AND b.symbol='USDT_IRT'
WHERE a.symbol='USD' AND a.close_price>0
GROUP BY left(a.date,4)
ORDER BY left(a.date,4);
SELECT 'adjusted_fx_join', left(a.date_time,4), count(*)::text,
       count(f.id)::text, (count(*)-count(f.id))::text
FROM marketdata_marketcandle a
LEFT JOIN marketdata_goldcurrencyhistory f ON f.symbol='USD' AND f.date=a.date_time
WHERE a.timeframe='1d_adj' AND a.date_time>='1400-01-01'
GROUP BY left(a.date_time,4)
ORDER BY left(a.date_time,4);
SELECT 'corporate_actions', source, kind, count(*)::text, count(DISTINCT symbol)::text
FROM marketdata_corporateaction
GROUP BY source, kind ORDER BY count(*) DESC;
''')
for row in checks:
    print(" | ".join(row))


## Dated interpretation

At the audit point, cash USD had 5,407 unique dates (1390-09-05–1405-07-02) and USDT/IRT had 2,780 (1397-09-06–1405-07-03), both positive and labeled Toman. The uniqueness constraint on `(symbol,date)` matched the distinct-date counts. Cash USD had ten missing calendar dates, all in the current Jalali year; USDT/IRT had 80 missing dates over its full span. Cash USD's longest unchanged-price run was 16 dates (1404-10-18–1404-11-03), including nine فولاد trading sessions. This is a flag for source investigation, not proof the underlying cash market traded on those days.

Cash USD's historical provider identity cannot be reconstructed per row: 5,361 of 5,407 rows lack `ingested_at` and `last_correlation_id`. The USDT rows are a different economic instrument; their same-date median absolute gap against cash USD ranged from 0.44% to 1.91% across overlapping Jalali years, with yearly p90 as high as 6.06%. A single-symbol adjusted-candle sample (فولاد) and the all-symbol adjusted-candle join since 1400 found no missing same-date cash-USD row, but that establishes join coverage only, not adjusted-return validity. The corporate-action ledger starts in 1400/1401 and cannot independently justify all earlier provider-adjusted candles.

**Decision:** keep original Rial stock prices and Toman cash-USD rates as separate observations. Withhold historical USD returns until the provider/source lineage, unusual unchanged-rate intervals, adjusted-price factors and dividend handling are resolved. Never fill a cash-USD gap from USDT without an explicit, user-visible alternative-basis definition.
